In [1]:
import pandas as pd

#Concatenação dos dfs de input
dfs = []
anos_copa = list(set(pd.read_csv('dados/copas/FIFA - World Cup Summary.csv')['YEAR']))

for ano in anos_copa:
    dfs.append(pd.read_csv(f'dados/dados_modelo/input_modelo{ano}.csv'))

df = pd.concat(dfs, ignore_index=True)
display(df)

,wins_difference,draws_difference,loses_difference,points_per_game_difference,result,goal_diff_pg_difference
0,0.166667,0.125000,-0.291667,0.62,1,1.96
1,0.000000,0.222222,-0.222222,0.22,1,-0.71
2,0.444444,0.055556,0.500000,NaN,1,NaN
3,0.378788,0.090909,-0.469697,1.23,1,3.73
4,0.393333,0.155000,-0.548333,1.34,1,4.00
...,...,...,...,...,...,...
975,0.014406,-0.048820,0.034414,0.00,2,0.49
976,0.170204,0.035510,-0.205714,0.55,1,1.05
977,-0.007347,0.004898,0.002449,-0.02,1,0.06
978,-0.150204,-0.015510,0.165714,-0.47,1,-0.81


In [2]:
from sklearn.model_selection import cross_validate, TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np
import xgboost as xgb


X = df.drop(columns=['result']).fillna(0)
Y = df["result"]
#Teste usando dados de 1930 á 2022, porém sem usar ranking da fifa

#Definição do Divisor Temporal
tscv = TimeSeriesSplit(n_splits=5)

# Modelos
modelos = {
    "Logistic Regression (Normalizada)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, class_weight="balanced", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
    "XGBoost (Desafiante)": xgb.XGBClassifier(
        n_estimators=100,           
        max_depth=3,                
        learning_rate=0.1,
        objective='multi:softprob', 
        num_class=3,
        eval_metric='mlogloss',    
        random_state=42
    )
}

#As 3 Métricas que queremos avaliar ao mesmo tempo
metricas = ['accuracy', 'roc_auc_ovr', 'neg_log_loss']

for nome, modelo in modelos.items():

    resultados = cross_validate(modelo, X, Y, cv=tscv, scoring=metricas)
    
    acc_media = np.mean(resultados['test_accuracy']) * 100
    acc_desvio = np.std(resultados['test_accuracy']) * 100 
    roc_media = np.mean(resultados['test_roc_auc_ovr'])
    logloss_media = -np.mean(resultados['test_neg_log_loss'])
    
    print(f"\n {nome}:")
    print(f"  Acurácia:  {acc_media:.2f}% (± {acc_desvio:.2f}%)")
    print(f"  ROC-AUC:   {roc_media:.4f}  (Quanto mais próximo de 1.0, melhor)")
    print(f"  Log Loss:  {logloss_media:.4f} (Quanto mais próximo de 0.0, melhor)")


 Logistic Regression (Normalizada):
  Acurácia:  49.45% (± 9.74%)
  ROC-AUC:   0.5540  (Quanto mais próximo de 1.0, melhor)
  Log Loss:  1.1965 (Quanto mais próximo de 0.0, melhor)

 Random Forest:
  Acurácia:  43.31% (± 3.28%)
  ROC-AUC:   0.5478  (Quanto mais próximo de 1.0, melhor)
  Log Loss:  1.6916 (Quanto mais próximo de 0.0, melhor)

 Gradient Boosting:
  Acurácia:  49.20% (± 9.36%)
  ROC-AUC:   0.5515  (Quanto mais próximo de 1.0, melhor)
  Log Loss:  1.5123 (Quanto mais próximo de 0.0, melhor)

 XGBoost (Desafiante):
  Acurácia:  49.94% (± 8.99%)
  ROC-AUC:   0.5569  (Quanto mais próximo de 1.0, melhor)
  Log Loss:  1.2670 (Quanto mais próximo de 0.0, melhor)


In [3]:
"""Os rankings da fifa não foram usados, e eles são um ótimo meio 
de avaliação. Eles surgiram em 1994, justamente na época 'moderna' do futebol...
Iremos testar se usar os dados de 94 em diante, porém com os dados dos rankings da fifa,
servirão para aumentar a precisão do modelo..."""

"Os rankings da fifa não foram usados, e eles são um ótimo meio \nde avaliação. Eles surgiram em 1994, justamente na época 'moderna' do futebol...\nIremos testar se usar os dados de 94 em diante, porém com os dados dos rankings da fifa,\nservirão para aumentar a precisão do modelo..."

In [4]:
import pandas as pd
import numpy as np
from normalize_countries import normalize_team_name

# Para isso, teremos que refazer os csv novamente
anos_copa = [i for i in range(1994, 2023, 4)]
dfs = []

for ano in anos_copa:
    df_confrontos = pd.read_csv(f"dados/confrontos_copas/confrontos{ano}.csv")
    df_ciclo = pd.read_csv(f"dados/estatisticas_ciclos/estatisticas_ciclo{ano}.csv")
    df_ranking = pd.read_csv(f"dados/rankings/fifa_pre_copa/fifa_ranking_pre_copa_{ano}.csv")
    
    # Preparar Ranking
    df_ranking["country_full"] = df_ranking["country_full"].apply(normalize_team_name)
    df_ranking = df_ranking.set_index("country_full")
    df_ranking = df_ranking[~df_ranking.index.duplicated(keep='first')]
    
    # Preparar Estatísticas do Ciclo
    df_ciclo["team"] = df_ciclo["team"].apply(normalize_team_name)
    df_ciclo = df_ciclo.set_index("team")
    df_ciclo["win_rate"] = (df_ciclo["wins"] / df_ciclo["games_played"]).fillna(0)
    df_ciclo["draw_rate"] = (df_ciclo["draws"] / df_ciclo["games_played"]).fillna(0)
    df_ciclo["lose_rate"] = (df_ciclo["loses"] / df_ciclo["games_played"]).fillna(0)
    
    df_confrontos.columns = df_confrontos.columns.str.lower().str.replace(' ', '_')
    
    wins_diff, draws_diff, loses_diff, ppg_diff, rank_diff, results, goal_diff_pg_diff = [], [], [], [], [], [], []

    
    for row in df_confrontos.itertuples():
        home_team = normalize_team_name(row.home_team_name)
        away_team = normalize_team_name(row.away_team_name)
        
        # Pular se não tivermos os dados das estatísticas do ciclo
        if home_team not in df_ciclo.index or away_team not in df_ciclo.index:
            continue
            
        #Posição ruim caso fora do ranking
        rank_home = df_ranking.loc[home_team, "rank"] if home_team in df_ranking.index else 100
        rank_away = df_ranking.loc[away_team, "rank"] if away_team in df_ranking.index else 100
        
    
        wins_diff.append(df_ciclo.loc[home_team]["win_rate"] - df_ciclo.loc[away_team]["win_rate"])
        draws_diff.append(df_ciclo.loc[home_team]["draw_rate"] - df_ciclo.loc[away_team]["draw_rate"])
        loses_diff.append(df_ciclo.loc[home_team]["lose_rate"] - df_ciclo.loc[away_team]["lose_rate"])
        ppg_diff.append(df_ciclo.loc[home_team]["points_per_game"] - df_ciclo.loc[away_team]["points_per_game"])
        goal_diff_pg_diff.append(df_ciclo.loc[home_team]["goal_diff_per_game"] - df_ciclo.loc[away_team]["goal_diff_per_game"])
        
        # Diferença de Ranking: Um valor NEGATIVO é bom para a equipe da Casa.
        # Exemplo: Brasil(1) - Japão(30) = -29. 
        rank_diff.append(rank_home - rank_away)
        results.append(row.resultado)
        
    # Montar o DataFrame do Ano
    df_ano = pd.DataFrame({
        "wins_difference": wins_diff,
        "draws_difference": draws_diff,
        "loses_difference": loses_diff,
        "points_per_game_difference": ppg_diff,
        "ranking_difference": rank_diff, 
        "goal_diff_per_game_difference": goal_diff_pg_diff,
        "result": results,
        
    })
    
    dfs.append(df_ano)
    print(f"Ano {ano}: Processado com sucesso! ({len(df_ano)} jogos)")


df = pd.concat(dfs, ignore_index=True)
df = df.fillna(0)
print(f"\n Jogos de 1994-2022, portando ranking da FIFA: {len(df)}")
display(df)


Ano 1994: Processado com sucesso! (52 jogos)
Ano 1998: Processado com sucesso! (64 jogos)
Ano 2002: Processado com sucesso! (64 jogos)
Ano 2006: Processado com sucesso! (64 jogos)
Ano 2010: Processado com sucesso! (64 jogos)
Ano 2014: Processado com sucesso! (80 jogos)
Ano 2018: Processado com sucesso! (64 jogos)
Ano 2022: Processado com sucesso! (64 jogos)

 Jogos de 1994-2022, portando ranking da FIFA: 516


,wins_difference,draws_difference,loses_difference,points_per_game_difference,ranking_difference,goal_diff_per_game_difference,result
0,-0.005348,-0.060606,0.065954,-0.07,-32.0,-0.18,0
1,0.380263,-0.113158,-0.267105,1.03,-42.0,1.15,1
2,-0.273913,0.075776,0.198137,-0.75,11.0,-1.20,0
3,0.001838,-0.011029,0.009191,0.00,-10.0,0.21,2
4,-0.150719,0.310819,-0.160100,-0.14,10.0,-0.16,2
...,...,...,...,...,...,...,...
511,0.014406,-0.048820,0.034414,0.00,1.0,0.49,2
512,0.170204,0.035510,-0.205714,0.55,-9.0,1.05,1
513,-0.007347,0.004898,0.002449,-0.02,-18.0,0.06,1
514,-0.150204,-0.015510,0.165714,-0.47,-10.0,-0.81,1


In [ ]:
from sklearn.model_selection import cross_validate, TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np
import xgboost as xgb

#É literalmente o mesmo código de antes, mas o df agora é diferente

X = df.drop(columns=['result']).fillna(0)
Y = df["result"]


#Teste usando dados de 1994 á 2022, USANDO RANKING DA FIFA

#Definição do Divisor Temporal
tscv = TimeSeriesSplit(n_splits=5)

# Modelos
modelos = {
    "Logistic Regression (Normalizada)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, class_weight="balanced", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
    "XGBoost (Desafiante)": xgb.XGBClassifier(
        n_estimators=100,           
        max_depth=3,                
        learning_rate=0.1,
        objective='multi:softprob', 
        num_class=3,
        eval_metric='mlogloss',     
        random_state=42
    )
}

#As 3 Métricas que queremos avaliar ao mesmo tempo
metricas = ['accuracy', 'roc_auc_ovr', 'neg_log_loss']

for nome, modelo in modelos.items():

    resultados = cross_validate(modelo, X, Y, cv=tscv, scoring=metricas)
    
    acc_media = np.mean(resultados['test_accuracy']) * 100
    acc_desvio = np.std(resultados['test_accuracy']) * 100 
    roc_media = np.mean(resultados['test_roc_auc_ovr'])
    logloss_media = -np.mean(resultados['test_neg_log_loss'])
    
    print(f"\n {nome}:")
    print(f"  Acurácia:  {acc_media:.2f}% (± {acc_desvio:.2f}%)")
    print(f"  ROC-AUC:   {roc_media:.4f}  (Quanto mais próximo de 1.0, melhor)")
    print(f"  Log Loss:  {logloss_media:.4f} (Quanto mais próximo de 0.0, melhor)")


 Logistic Regression (Normalizada):
  Acurácia:  52.33% (± 6.28%)
  ROC-AUC:   0.6308  (Quanto mais próximo de 1.0, melhor)
  Log Loss:  1.0152 (Quanto mais próximo de 0.0, melhor)

 Random Forest:
  Acurácia:  45.35% (± 5.45%)
  ROC-AUC:   0.6219  (Quanto mais próximo de 1.0, melhor)
  Log Loss:  1.0575 (Quanto mais próximo de 0.0, melhor)

 Gradient Boosting:
  Acurácia:  40.93% (± 6.17%)
  ROC-AUC:   0.5956  (Quanto mais próximo de 1.0, melhor)
  Log Loss:  1.3352 (Quanto mais próximo de 0.0, melhor)

 XGBoost (Desafiante):
  Acurácia:  45.12% (± 5.99%)
  ROC-AUC:   0.6082  (Quanto mais próximo de 1.0, melhor)
  Log Loss:  1.1875 (Quanto mais próximo de 0.0, melhor)


In [6]:
""" 
Embora a Regressão Logística tenha apresentado uma leve vantagem nas métricas do modelo base (baseline), 
a escolha definitiva para a arquitetura preditiva deste projeto recai sobre o XGBoost, sustentada por dois pilares 
técnicos essenciais:

1. Tratamento de Não-linearidade e Interação de Variáveis
O futebol é um esporte de dinâmica complexa onde as variáveis não atuam de forma isolada ou linear. 
A Regressão Logística assume que o impacto das variáveis é independente e constante (ex: assumindo que uma 
diferença de -20 no ranking tem exatamente o dobro do peso de uma diferença de -10, independentemente do contexto). 
O XGBoost, por ser um modelo ensemble baseado em árvores de decisão, lida nativamente com a não-linearidade. 
Ele consegue cruzar variáveis e atribuir "pesos dinâmicos" 
(ex: a diferença de ranking só terá um peso altíssimo se o histórico recente de pontos por jogo também for favorável), 
capturando a verdadeira malícia e o contexto de uma partida.

2. Potencial de Otimização e Imunidade à Multicolinearidade
Nos testes preliminares, a Regressão Logística já operou muito próxima do seu limite teórico. 
O XGBoost, por outro lado, possui uma arquitetura robusta com diversos hiperparâmetros ajustáveis 
(como profundidade da árvore e taxa de aprendizado), oferecendo um teto de melhoria (potencial) 
muito maior após o Hyperparameter Tuning. Além disso, o XGBoost é naturalmente menos sensível à multicolinearidade. 
Como várias estatísticas esportivas trazem informações sobrepostas (ex: taxa de vitórias e média de pontos), 
a Regressão Logística pode sofrer instabilidade. O XGBoost ignora essas redundâncias perfeitamente, 
o que torna o nosso modelo altamente escalável para a adição de novas features no futuro sem prejudicar 
a calibração das probabilidades.


"""

' \nEmbora a Regressão Logística tenha apresentado uma leve vantagem nas métricas do modelo base (baseline), \na escolha definitiva para a arquitetura preditiva deste projeto recai sobre o XGBoost, sustentada por dois pilares \ntécnicos essenciais:\n\n1. Tratamento de Não-linearidade e Interação de Variáveis\nO futebol é um esporte de dinâmica complexa onde as variáveis não atuam de forma isolada ou linear. \nA Regressão Logística assume que o impacto das variáveis é independente e constante (ex: assumindo que uma \ndiferença de -20 no ranking tem exatamente o dobro do peso de uma diferença de -10, independentemente do contexto). \nO XGBoost, por ser um modelo ensemble baseado em árvores de decisão, lida nativamente com a não-linearidade. \nEle consegue cruzar variáveis e atribuir "pesos dinâmicos" \n(ex: a diferença de ranking só terá um peso altíssimo se o histórico recente de pontos por jogo também for favorável), \ncapturando a verdadeira malícia e o contexto de uma partida.\n\n2. 

In [15]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import numpy as np


X = df.drop(columns=['result']).fillna(0)
Y = df["result"]

X.to_csv("dados/X.csv")
Y.to_csv("dados/Y.csv")

tscv = TimeSeriesSplit(n_splits=5)


xgb_base = xgb.XGBClassifier(
    objective='multi:softprob', 
    num_class=3,
    eval_metric='mlogloss', 
    random_state=42
)

#Hiperparâmetros a serem testados
param_grid = {
    'max_depth': [3, 4],                # Quão complexa cada árvore pode ser
    'learning_rate': [0.01, 0.05],      # Velocidade de aprendizagem
    'n_estimators': [100, 200, 400],    # Como a taxa de aprendizagem é baixa (0.01), ele precisa de mais árvores!
    'subsample': [0.8, 0.75, 1],        # % de dados usados por árvore
    'reg_lambda': [1, 5, 10],           # Punição L2 (Quanto maior, mais conservador)
    'gamma': [0, 1, 5]                  # Exige uma redução de erro mínima para criar novos ramos
}

grid_search = GridSearchCV(
    estimator=xgb_base, 
    param_grid=param_grid, 
    cv=tscv, 
    scoring='neg_log_loss', #Critério que mais se aproxima de achar as probabilidades por time
    verbose=1, 
    n_jobs=-1 
)

grid_search.fit(X, Y)

print(f"Melhor Log Loss Encontrado: {-grid_search.best_score_:.4f}") 
print(f"HiperParâmetros Com melhor perfomance: {grid_search.best_params_}")

# Guardamos o modelo perfeito para usarmos nas previsões finais
melhor_xgb = grid_search.best_estimator_

grid_search = GridSearchCV(
    estimator=xgb_base, 
    param_grid=param_grid, 
    cv=tscv, 
    scoring='accuracy', #Testando acurácia para comparações
    verbose=1, 
    n_jobs=-1 
)

grid_search.fit(X, Y)

print(f"Melhor Acurácia Encontrada: {grid_search.best_score_ * 100:.2f}") 
print(f"HiperParâmetros Com melhor perfomance: {grid_search.best_params_}")

melhor_xgb2 = grid_search.best_estimator_

Fitting 5 folds for each of 324 candidates, totalling 1620 fits


KeyboardInterrupt: 

In [8]:
"""Adicionando um calibrador de probabilidades (XGboost tende a ser extremo demais nas probabilidades)"""

from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import cross_validate

# Pegamos o melhor xgboost encontrado (usamos o tscv no cross validate)
xgb_calibrado = CalibratedClassifierCV(melhor_xgb, method='sigmoid', cv=2)

# Vamos testá-lo novamente
resultados_calibrados = cross_validate(
    xgb_calibrado, X, Y, 
    cv=tscv, 
    scoring=['accuracy', 'neg_log_loss']
)

novo_logloss = -np.mean(resultados_calibrados['test_neg_log_loss'])
nova_acuracia = np.mean(resultados_calibrados['test_accuracy']) * 100

print(f"Log Loss do XGBoost Calibrado: {novo_logloss:.4f}")
print(f" Acurácia: {nova_acuracia:.2f}%")

Log Loss do XGBoost Calibrado: 1.0341
 Acurácia: 50.47%


In [17]:
"""O Melhor modelo foi a Logistic Regression por muito pouco, mas o XGBoost tem perfomance e 
praticidade maior do que o do LR para conseguir as estatísticas"""


def prever_jogo(ano_copa, time_casa, time_fora, modelo, X_treino):
    """
    Prevê o resultado de um jogo com base nas estatísticas e ranking de um determinado ano.
    Requer o modelo treinado e o DataFrame X usado no treino para alinhar as colunas.
    Pelos Modelos se sairem melhor com as features de ranking, a função só funciona a partir 
    da copa de 1994
    """

    # Garantir que os nomes estão normalizados
    time_casa_norm = normalize_team_name(time_casa)
    time_fora_norm = normalize_team_name(time_fora)
    
    try:
        #Preparando os df e normalizando os nomes 
        df_ciclo = pd.read_csv(f"dados/estatisticas_ciclos/estatisticas_ciclo{ano_copa}.csv")
        df_ranking = pd.read_csv(f"dados/rankings/fifa_pre_copa/fifa_ranking_pre_copa_{ano_copa}.csv")
        
        df_ciclo["team"] = df_ciclo["team"].apply(normalize_team_name)
        df_ciclo = df_ciclo.set_index("team")
        
        df_ranking["country_full"] = df_ranking["country_full"].apply(normalize_team_name)
        df_ranking = df_ranking.set_index("country_full")
        df_ranking = df_ranking[~df_ranking.index.duplicated(keep='first')] # Remove duplicados por segurança

        #Tratamento caso coloque um time que não vai para a copa
        if time_casa_norm not in df_ciclo.index:
            return f" Erro: {time_casa} não foi encontrado no arquivo de estatísticas de {ano_copa}."
        if time_fora_norm not in df_ciclo.index:
            return f" Erro: {time_fora} não foi encontrado no arquivo de estatísticas de {ano_copa}."

        #Calculando as estatísticas
        win_rate_casa = (df_ciclo.loc[time_casa_norm, "wins"] / df_ciclo.loc[time_casa_norm, "games_played"])
        win_rate_fora = (df_ciclo.loc[time_fora_norm, "wins"] / df_ciclo.loc[time_fora_norm, "games_played"])
        
        draw_rate_casa = (df_ciclo.loc[time_casa_norm, "draws"] / df_ciclo.loc[time_casa_norm, "games_played"])
        draw_rate_fora = (df_ciclo.loc[time_fora_norm, "draws"] / df_ciclo.loc[time_fora_norm, "games_played"])
        
        lose_rate_casa = (df_ciclo.loc[time_casa_norm, "loses"] / df_ciclo.loc[time_casa_norm, "games_played"])
        lose_rate_fora = (df_ciclo.loc[time_fora_norm, "loses"] / df_ciclo.loc[time_fora_norm, "games_played"])

        # Buscar Rankings (se não achar, assume rank 100 como punição neutra)
        rank_casa = df_ranking.loc[time_casa_norm, "rank"] if time_casa_norm in df_ranking.index else 100
        rank_fora = df_ranking.loc[time_fora_norm, "rank"] if time_fora_norm in df_ranking.index else 100

        #Organizando as diferenças
        dados_calculados = {
            "wins_difference": win_rate_casa - win_rate_fora,
            "draws_difference": draw_rate_casa - draw_rate_fora,
            "loses_difference": lose_rate_casa - lose_rate_fora,
            "points_per_game_difference": df_ciclo.loc[time_casa_norm, "points_per_game"] - df_ciclo.loc[time_fora_norm, "points_per_game"],
            "ranking_difference": rank_casa - rank_fora,
            "goal_diff_pg_difference": df_ciclo.loc[time_casa_norm, "goal_diff_per_game"] - df_ciclo.loc[time_fora_norm, "goal_diff_per_game"],
            "goal_diff_per_game_difference": df_ciclo.loc[time_casa_norm, "goal_diff_per_game"] - df_ciclo.loc[time_fora_norm, "goal_diff_per_game"]
        }

        #  Montar o DataFrame de input filtrando e ORDENANDO exatamente igual ao X de treino
        colunas_do_modelo = X_treino.columns.tolist()
        input_modelo = pd.DataFrame([{col: dados_calculados.get(col, 0) for col in colunas_do_modelo}])
        input_modelo = input_modelo.fillna(0)

        #  Executar a previsão de probabilidades
        probabilidades = modelo.predict_proba(input_modelo)[0]
        
        # Mapeamento dinâmico baseado nas classes do seu label /Y
        # Assume o padrão clássico do dataset: 0 para Vitória Casa, 1 para Empate, 2 para Vitória Fora
        # Se as suas classes forem strings (ex: 'Home', 'Draw'), o modelo.classes_ resolve automaticamente
        classes = modelo.classes_
        resultado_dict = dict(zip(classes, probabilidades))
        
        # Identificar quais posições representam cada resultado (ajuste os índices se necessário)
        prob_casa = resultado_dict.get(0, resultado_dict.get('Home', 0)) * 100
        prob_empate = resultado_dict.get(1, resultado_dict.get('Draw', 0)) * 100
        prob_fora = resultado_dict.get(2, resultado_dict.get('Away', 0)) * 100

        # 8. Exibir o resultado estilizado
        print(f"Copa do Mundo {ano_copa} | Previsão de Confronto:")
        print(f"{time_casa} vs {time_fora}")
        print("-" * 50)
        print(f" Vitória do(a) {time_casa}: {prob_casa:.2f}%")
        print(f" Empate: {prob_empate:.2f}%")
        print(f" Vitória do(a) {time_fora}: {prob_fora:.2f}%")
        print("-" * 50)
        
    except FileNotFoundError as e:
        print(f"Erro de arquivo: Verifique se os caminhos e nomes dos CSVs de {ano_copa} estão corretos.")
        print(f"Detalhe: {e}")

prever_jogo(2026, "Brasil", "França", melhor_xgb, X)

Copa do Mundo 2026 | Previsão de Confronto:
Brasil vs França
--------------------------------------------------
 Vitória do(a) Brasil: 28.99%
 Empate: 18.39%
 Vitória do(a) França: 52.62%
--------------------------------------------------
